In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
import time
from bs4 import BeautifulSoup
from datetime import datetime
import pandas as pd
from selenium.webdriver.support.ui import Select
from selenium.common.exceptions import StaleElementReferenceException
from deep_translator import GoogleTranslator

In [ ]:
# translation

def translate_text(text, source='auto', target='fr'):
    try:
        return GoogleTranslator(source=source, target=target).translate(text)
    except Exception as e:
        print("Translation error:", e)
        return text

In [ ]:
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"

options = webdriver.ChromeOptions()
options.add_argument(f"user-agent={user_agent}")

driver = webdriver.Chrome(options=options)

driver.get("LINK") #add link
time.sleep(3)

In [ ]:
# scraping

tenders=[]
wait = WebDriverWait(driver, 10)

back_counter = 0
max_pages = 100
page_count = 1

while page_count <= max_pages:
    rows = driver.find_elements(By.CSS_SELECTOR, "tr.mat-row")
    
    for i in range(len(rows)): 
        rows = driver.find_elements(By.CSS_SELECTOR, "tr.mat-row")
        row = rows[i]
        cells = row.find_elements(By.TAG_NAME, "td")
        
        if len(cells) >= 5:
            expiration_date = cells[4].text.strip()
            expiration_date = datetime.strptime(expiration_date, "%d/%m/%Y %H:%M")
            
            if expiration_date >= datetime.now():
                title = cells[3].text.strip()
                title = translate_text(title).lower()
                
                publish_date =  cells[2].text.strip()
                
                id_cell = row.find_element(By.CSS_SELECTOR, "td.mat-column-epBidMasterId")
                id1 = id_cell.get_attribute("textContent").strip()
                id2 = cells[0].text.strip()

                link= f"https://www.tuneps.tn/portail/offres/details/{id2}/{id1}"

                tenders.append({
                    "title": title,
                    "publication_date": publish_date,
                    "expiration_date": expiration_date.strftime("%d-%m-%Y"),
                    "link": link,
                    "country": "Tunisia"
                })

                back_counter= back_counter+1
                if back_counter == 10:
                    break
    try:

        if page_count >= max_pages:
            print(f"Limit of {max_pages} pages reached.")
            break
        btn_next = wait.until(EC.element_to_be_clickable(
            (By.CSS_SELECTOR, "button.mat-paginator-navigation-next")
        ))

        if "mat-button-disabled" in btn_next.get_attribute("class"):
            print("Last page reached.")
            break

        btn_next.click()
        page_count += 1  

        time.sleep(1.5)
        wait.until(EC.staleness_of(rows[0]))
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "tr.mat-row")))

    except Exception as e:
        print("No next button or error:", e)
        break

In [ ]:
driver.quit()

In [ ]:
df = pd.DataFrame(tenders)
df.to_excel("../2_scraped_files/Tunisia.xlsx", index=False) # modify